# Phase 2: GNN Architecture Prototyping

**Objective.** Demonstrate, on a single target ADR, the text-attributed heterogeneous graph neural network used in this thesis. This notebook is intentionally small-scale and meant for inspection and prototyping; the full multi-ADR training pipeline is in `scripts/phase2_gnn/train_job.py`.

**Pipeline shown here:**
1. Load the unified heterogeneous graph (`Data/MTP_Graph.pt`) produced by Phase 1.
2. Inject ClinicalBERT-encoded drug features (`Data/real_drug_features.pt`) into the `drug` node type.
3. Purge non-target `causes_*` edge types (matching the training-time protocol).
4. Define the encoder (`HeteroADRModel`) and dot-product decoder (`LinkPredictor`).
5. Run a short training loop on one target ADR (`thrombocytopenia` by default) using `LinkNeighborLoader`-based mini-batching.

**For the full multi-ADR pipeline** (50 epochs per ADR, four ADRs in parallel on HPC), see `scripts/phase2_gnn/train_job.py` and `hpc/train_adr.sbatch`.

**Note on bug fixes.** Two subtle issues affect any model loading: (1) the encoder must be built with the *purged* edge-type metadata (not the full graph), to match the saved checkpoint structure; (2) PyG `Linear(-1, h)` lazy layers must be materialized via a dummy forward pass *before* `load_state_dict`. Both fixes are demonstrated here and in `scripts/phase2_gnn/evaluate_all_adrs.py`.

## Step 1: Imports and graph loading

In [ ]:
import os
import time
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv, Linear
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score

print(f'PyTorch version: {torch.__version__}')

# 1. Load the heterogeneous graph constructed in Phase 1.
GRAPH_PATH = '../Data/MTP_Graph.pt'
FEATURE_PATH = '../Data/real_drug_features.pt'

assert os.path.exists(GRAPH_PATH), f'Run Phase 1 first to create {GRAPH_PATH}'
assert os.path.exists(FEATURE_PATH), f'Run feature_encoder.py first to create {FEATURE_PATH}'

data = torch.load(GRAPH_PATH, weights_only=False)
print(f'Graph loaded: {data}')
print(f'  Node types: {len(data.node_types)}')
print(f'  Edge types: {len(data.edge_types)}')

# 2. Inject ClinicalBERT drug features.
real_drug_features = torch.load(FEATURE_PATH, weights_only=False)
data['drug'].x = real_drug_features
print(f'Injected ClinicalBERT features into drug nodes: shape={real_drug_features.shape}')

## Step 2: Select target ADR and purge non-target edges

The training script removes all `causes_*` edge types except the target before building the model. This keeps the encoder size manageable (51 edge types instead of 1{,}359) and prevents shortcuts through the multiplex polypharmacy layer.

In [ ]:
TARGET_ADR = 'thrombocytopenia'   # change to Bleeding / Cardiacdecompensation / kidneyfailure if desired
target_edge_type = ('drug', f'causes_{TARGET_ADR}', 'drug')

# Verify the target exists in the graph.
assert target_edge_type in data.edge_types, (
    f'{target_edge_type} not in graph. '
    f'Sample causes_* edges: '
    f'{[et for et in data.edge_types if et[1].startswith("causes_")][:5]}'
)
n_pos = data[target_edge_type].edge_index.size(1)
print(f'Target: {target_edge_type}')
print(f'Positive edges: {n_pos}')

# Build the purged HeteroData: keep all node types, all structural edges,
# and only the single target causes_* edge type.
data_purged = HeteroData()
for nt in data.node_types:
    if hasattr(data[nt], 'x') and data[nt].x is not None:
        data_purged[nt].x = data[nt].x
    data_purged[nt].num_nodes = data[nt].num_nodes

purged_count = 0
for et in data.edge_types:
    if et[1].startswith('causes_') and et != target_edge_type:
        purged_count += 1
        continue
    data_purged[et].edge_index = data[et].edge_index

print(f'Purged {purged_count} non-target ADR edge types.')
print(f'Kept {len(data_purged.edge_types)} edge types '
      f'(50 structural + 1 target).')

## Step 3: Encoder and decoder definitions

**Encoder** (`HeteroADRModel`): per-node-type linear input projection $\mathbf{W}^{(\text{in})}_{\tau}$, two `HeteroConv` layers (one `SAGEConv` per relation, mean aggregation), ReLU and dropout between, and a final linear output projection $\mathbf{W}^{(\text{out})}$.

**Decoder** (`LinkPredictor`): parameter-free dot product of source and destination drug embeddings.

This matches `scripts/phase2_gnn/train_job.py` exactly.

In [ ]:
class HeteroADRModel(torch.nn.Module):
    """Two-layer heterogeneous GNN encoder."""
    def __init__(self, metadata, hidden_channels=128, out_channels=64):
        super().__init__()
        # Per-node-type input projections (lazy: dim inferred on first forward)
        self.lin_dict = torch.nn.ModuleDict({
            nt: Linear(-1, hidden_channels) for nt in metadata[0]
        })
        # Two HeteroConv layers, each with SAGEConv per relation type
        self.convs = torch.nn.ModuleList([
            HeteroConv(
                {et: SAGEConv((-1, -1), hidden_channels) for et in metadata[1]},
                aggr='mean'
            )
            for _ in range(2)
        ])
        self.lin_out = Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        x_dict = {k: self.lin_dict[k](x) for k, x in x_dict.items()}
        x_dict = self.convs[0](x_dict, edge_index_dict)
        x_dict = {k: F.relu(x) for k, x in x_dict.items()}
        x_dict = {k: F.dropout(x, p=0.3, training=self.training) for k, x in x_dict.items()}
        x_dict = self.convs[1](x_dict, edge_index_dict)
        return {k: self.lin_out(x) for k, x in x_dict.items()}


class LinkPredictor(torch.nn.Module):
    """Parameter-free dot-product decoder."""
    def __init__(self):
        super().__init__()

    def forward(self, z_src, z_dst):
        return (z_src * z_dst).sum(dim=-1)


print('Architecture classes defined.')

## Step 4: Train/val/test split and mini-batch loaders

Uses `RandomLinkSplit` (80/10/10, undirected) and `LinkNeighborLoader` for neighbor sampling. The full pipeline uses `batch_size=1024` and `num_neighbors=[20, 10]`; here we use a smaller scale for laptop prototyping.

In [ ]:
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,                # Symmetric treatment of (u,v) and (v,u)
    add_negative_train_samples=True,    # Negative sampling within the splitter
    neg_sampling_ratio=1.0,             # One negative per positive
    edge_types=target_edge_type,
)
train_data, val_data, test_data = transform(data_purged)
print('Split completed.')

# Loaders --- small scale for prototyping. The HPC training script uses
# BATCH_SIZE=1024 and NUM_NEIGHBORS=[20, 10].
BATCH_SIZE = 64
NUM_NEIGHBORS = [10, 5]

train_loader = LinkNeighborLoader(
    train_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    edge_label_index=(target_edge_type, train_data[target_edge_type].edge_label_index),
    edge_label=train_data[target_edge_type].edge_label, shuffle=True,
)
val_loader = LinkNeighborLoader(
    val_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    edge_label_index=(target_edge_type, val_data[target_edge_type].edge_label_index),
    edge_label=val_data[target_edge_type].edge_label, shuffle=False,
)
test_loader = LinkNeighborLoader(
    test_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    edge_label_index=(target_edge_type, test_data[target_edge_type].edge_label_index),
    edge_label=test_data[target_edge_type].edge_label, shuffle=False,
)
print(f'Loaders ready. Batch size={BATCH_SIZE}, neighbors={NUM_NEIGHBORS}.')

## Step 5: Short training loop (5 epochs)

A short prototyping run to validate end-to-end correctness. For the full 50-epoch multi-ADR pipeline, use `scripts/phase2_gnn/train_job.py`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

encoder = HeteroADRModel(data_purged.metadata(),
                          hidden_channels=128, out_channels=64).to(device)
decoder = LinkPredictor().to(device)

optimizer = optim.Adam(list(encoder.parameters()), lr=1e-3)
loss_fn = torch.nn.BCEWithLogitsLoss()


def train_epoch():
    encoder.train()
    total_loss, n_samples = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        z = encoder(batch.x_dict, batch.edge_index_dict)
        links = batch[target_edge_type]
        src, dst = links.edge_label_index[0], links.edge_label_index[1]
        scores = decoder(z['drug'][src], z['drug'][dst])
        loss = loss_fn(scores, links.edge_label.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * src.size(0)
        n_samples += src.size(0)
    return total_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate(loader):
    encoder.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        z = encoder(batch.x_dict, batch.edge_index_dict)
        links = batch[target_edge_type]
        src, dst = links.edge_label_index[0], links.edge_label_index[1]
        scores = decoder(z['drug'][src], z['drug'][dst])
        all_preds.append(scores.sigmoid().cpu())
        all_labels.append(links.edge_label.cpu())
    return roc_auc_score(
        torch.cat(all_labels).numpy(),
        torch.cat(all_preds).numpy(),
    )


print('Starting 5-epoch prototyping run...')
for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch()
    val_auc = evaluate(val_loader)
    print(f'Epoch {epoch:02d} | Loss {loss:.4f} | Val AUC {val_auc:.4f} | Time {time.time()-t0:.1f}s')

test_auc = evaluate(test_loader)
print(f'\nFinal test AUC: {test_auc:.4f}')
print('Prototyping complete. For full-scale training, use scripts/phase2_gnn/train_job.py.')

## Next steps

1. **Full multi-ADR training:** see `scripts/phase2_gnn/train_job.py` (and `hpc/train_adr.sbatch` for SLURM submission).
2. **Multi-ADR evaluation under both protocols:** `scripts/phase2_gnn/evaluate_all_adrs.py`.
3. **Counterfactual edge-occlusion explainability:** `scripts/phase3_explainability/phase3_subgraph_viz.py`.

See `README.md` for full reproduction commands.